# Uniformity Principle Readability Experiment Demo

This notebook demonstrates the Uniformity Principle for sentence-level readability prediction. 
The experiment compares three feature sets:
- **Average-only**: Traditional readability features (word length, syllables, frequency)
- **Uniformity-only**: Coefficient of variation within sentences (how uniform the features are)
- **Combined**: Both average and uniformity features

## Key Finding
Uniformity features provide significant additional predictive power beyond average features alone.

## Dataset
This demo uses a curated subset of the WeeBIT and CEFR-SP datasets with diverse examples.

In [ ]:
# Install dependencies
import subprocess, sys
def _pip(*a): subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', *a])

# loguru is NOT pre-installed on Colab, always install
_pip('loguru')

# Core packages (pre-installed on Colab, install locally to match Colab env)
if 'google.colab' not in sys.modules:
    _pip('numpy==2.0.2', 'pandas==2.2.2', 'scikit-learn==1.6.1', 'scipy==1.16.3', 'matplotlib==3.10.0')

In [ ]:
# Imports - copied from original script
from loguru import logger
from pathlib import Path
import json
import sys
import numpy as np
from sklearn.linear_model import Ridge
from sklearn.model_selection import KFold, cross_val_score
from sklearn.preprocessing import StandardScaler
import warnings
warnings.filterwarnings('ignore')

# Additional imports for the notebook
import matplotlib.pyplot as plt
from sklearn.metrics import r2_score

In [ ]:
# Data loading helper - uses GitHub URL with local fallback
GITHUB_DATA_URL = "https://raw.githubusercontent.com/AMGrobelnik/ai-invention-fea63a-the-uniformity-principle-how-within-sent/main/round-2/experiment-1/demo/mini_demo_data.json"

def load_data():
    """Load data from GitHub URL with local fallback."""
    try:
        import urllib.request
        with urllib.request.urlopen(GITHUB_DATA_URL) as response:
            return json.loads(response.read().decode())
    except Exception:
        pass
    if os.path.exists("mini_demo_data.json"):
        with open("mini_demo_data.json") as f:
            return json.load(f)
    raise FileNotFoundError("Could not load mini_demo_data.json")

import os

In [ ]:
## Configuration

Set tunable parameters for the experiment. For this demo with 12 examples:
- `N_BOOTSTRAP`: Number of bootstrap samples (original: 2000, demo: 200)
- `CV_SPLITS`: Number of cross-validation folds (original: 5, demo: 2 - adjusted for small dataset)
- `RIDGE_ALPHA`: Regularization strength for Ridge regression (original: 1.0)

# Configuration - scaled for demo with 12 examples
# Original values commented out for reference
N_BOOTSTRAP = 200  # Original: 2000 - increased for more stable CI
CV_SPLITS = 2      # Original: 5 - adjusted for small dataset (12 examples)
RIDGE_ALPHA = 1.0  # Original:1.0 - unchanged

print(f"Configuration:")
print(f"  N_BOOTSTRAP: {N_BOOTSTRAP}")
print(f"  CV_SPLITS: {CV_SPLITS}")
print(f"  RIDGE_ALPHA: {RIDGE_ALPHA}")

In [ ]:
# Configuration - ABSOLUTE MINIMUM values for demo
N_BOOTSTRAP = 100  # Original: 2000 - reduced for faster demo
CV_SPLITS = 3      # Original: 5 - reduced for small dataset
RIDGE_ALPHA = 1.0  # Original: 1.0 - unchanged

print(f"Configuration:")
print(f"  N_BOOTSTRAP: {N_BOOTSTRAP}")
print(f"  CV_SPLITS: {CV_SPLITS}")
print(f"  RIDGE_ALPHA: {RIDGE_ALPHA}")

## Feature Extraction

Extract linguistic features from sentences:
1. **Average features**: Mean word length, syllables, word frequency, sentence length
2. **Uniformity features**: Coefficient of variation (CV) of word length, syllables, word frequency

The uniformity features capture how consistent these properties are within a sentence.

In [ ]:
# Feature extraction function - copied from original script
def extract_features(examples: list) -> tuple:
    """
    Extract features from examples.
    
    Features:
    0: avg_word_length
    1: avg_syllables (heuristic)
    2: avg_word_freq (heuristic - based on word length and common words)
    3: sentence_length
    4: cv_word_length (uniformity)
    5: cv_syllables (uniformity)
    6: cv_word_freq (uniformity)
    """
    features = []
    feature_names = ['avg_word_length', 'avg_syllables', 'avg_word_freq', 'sentence_length',
                     'cv_word_length', 'cv_syllables', 'cv_word_freq']
    
    # Common English words (higher frequency)
    common_words = set(['the', 'be', 'to', 'of', 'and', 'a', 'in', 'that', 'have', 'i',
                        'it', 'for', 'not', 'on', 'with', 'he', 'as', 'you', 'do', 'at',
                        'this', 'but', 'his', 'by', 'from', 'they', 'we', 'say', 'her', 'she',
                        'or', 'an', 'will', 'my', 'one', 'all', 'would', 'there', 'their', 'what'])
    
    for i, ex in enumerate(examples):
        if i % 10 == 0 and i > 0:
            print(f"  Extracted features for {i}/{len(examples)} examples")
        
        text = ex['input']
        words = text.split()
        
        if not words:
            features.append([0] * len(feature_names))
            continue
        
        # Word lengths
        word_lengths = [len(w) for w in words]
        
        # Syllable heuristic: count vowel groups
        syllable_counts = []
        for w in words:
            w_lower = w.lower()
            vowels = sum(1 for c in w_lower if c in 'aeiouy')
            # Rough heuristic
            syllables = max(1, vowels)
            syllable_counts.append(syllables)
        
        # Word frequency heuristic: common words = higher freq
        word_freqs = []
        for w in words:
            w_lower = w.lower().strip('.,!?;:"\'()[]{}')
            if w_lower in common_words:
                word_freqs.append(3.0)  # high frequency
            elif len(w_lower) <= 4:
                word_freqs.append(2.0)  # medium frequency
            else:
                word_freqs.append(1.0)  # low frequency
        word_freqs_log = [np.log(f + 1) for f in word_freqs]
        
        # Average features
        avg_word_length = np.mean(word_lengths)
        avg_syllables = np.mean(syllable_counts)
        avg_word_freq = np.mean(word_freqs_log)
        sentence_length = len(words)
        
        # Uniformity features (coefficient of variation)
        cv_word_length = np.std(word_lengths) / (avg_word_length + 1e-10)
        cv_syllables = np.std(syllable_counts) / (avg_syllables + 1e-10)
        cv_word_freq = np.std(word_freqs_log) / (avg_word_freq + 1e-10) if word_freqs_log else 0
        
        features.append([
            avg_word_length, avg_syllables, avg_word_freq, sentence_length,
            cv_word_length, cv_syllables, cv_word_freq
        ])
    
    return np.array(features), feature_names

# Extract features from all examples
print("Extracting features...")
X, feature_names = extract_features(all_examples)
y = np.array([float(ex['output']) for ex in all_examples])

print(f"\nFeature matrix shape: {X.shape}")
print(f"Target vector shape: {y.shape}")
print(f"\nFeature names: {feature_names}")
print(f"\nSample features (first example):")
for i, name in enumerate(feature_names):
    print(f"  {name}: {X[0, i]:.4f}")

## Model Evaluation

Evaluate three feature sets using cross-validation:
1. Average-only features (indices 0-3)
2. Uniformity-only features (indices 4-6)
3. Combined features (indices 0-6)

In [ ]:
# Evaluation functions - copied from original script
def evaluate_feature_set(X: np.ndarray, y: np.ndarray, feature_names: list,
                        feature_indices: list, cv: KFold) -> dict:
    """Evaluate a feature set using cross-validation."""
    X_subset = X[:, feature_indices]
    scaler = StandardScaler()
    X_scaled = scaler.fit_transform(X_subset)
    
    model = Ridge(alpha=RIDGE_ALPHA)
    scores_r2 = cross_val_score(model, X_scaled, y, cv=cv, scoring='r2')
    
    return {
        'feature_names': [feature_names[i] for i in feature_indices],
        'r2_mean': float(np.mean(scores_r2)),
        'r2_std': float(np.std(scores_r2)),
    }

# Define feature sets
avg_indices = [0, 1, 2, 3]  # avg features only
uniformity_indices = [4, 5, 6]  # CV features only
combined_indices = [0, 1, 2, 3, 4, 5, 6]  # all features

# Cross-validation setup
cv = KFold(n_splits=CV_SPLITS, shuffle=True, random_state=42)

# Evaluate each feature set
print("Evaluating average-only features...")
results_avg = evaluate_feature_set(X, y, feature_names, avg_indices, cv)

print("Evaluating uniformity-only features...")
results_uni = evaluate_feature_set(X, y, feature_names, uniformity_indices, cv)

print("Evaluating combined features...")
results_comb = evaluate_feature_set(X, y, feature_names, combined_indices, cv)

results = {
    'average_only': results_avg,
    'uniformity_only': results_uni,
    'combined': results_comb
}

print("\n" + "="*60)
print("CROSS-VALIDATION RESULTS")
print("="*60)
for method, result in results.items():
    print(f"{method}: R² = {result['r2_mean']:.4f} +/- {result['r2_std']:.4f}")

## Generate Predictions

Train models on all data and generate predictions for comparison.

In [ ]:
# Generate predictions function - copied from original script
def generate_predictions(X: np.ndarray, y: np.ndarray,
                        avg_indices: list, uniformity_indices: list, 
                        combined_indices: list) -> dict:
    """Generate predictions using all training data."""
    predictions = {}
    
    # Average-only
    scaler = StandardScaler()
    X_avg = scaler.fit_transform(X[:, avg_indices])
    model = Ridge(alpha=RIDGE_ALPHA)
    model.fit(X_avg, y)
    predictions['average_only'] = model.predict(X_avg)
    
    # Uniformity-only
    scaler = StandardScaler()
    X_uni = scaler.fit_transform(X[:, uniformity_indices])
    model = Ridge(alpha=RIDGE_ALPHA)
    model.fit(X_uni, y)
    predictions['uniformity_only'] = model.predict(X_uni)
    
    # Combined
    scaler = StandardScaler()
    X_comb = scaler.fit_transform(X[:, combined_indices])
    model = Ridge(alpha=RIDGE_ALPHA)
    model.fit(X_comb, y)
    predictions['combined'] = model.predict(X_comb)
    
    return predictions

# Generate predictions
print("Generating predictions...")
predictions = generate_predictions(X, y, avg_indices, uniformity_indices, combined_indices)

print("\nPredictions generated for all examples.")
print(f"Sample predictions (first 3 examples):")
for i in range(min(3, len(all_examples))):
    print(f"\nExample {i+1}:")
    print(f"  Text: {all_examples[i]['input'][:60]}...")
    print(f"  True score: {y[i]:.2f}")
    print(f"  Pred (avg-only): {predictions['average_only'][i]:.4f}")
    print(f"  Pred (uniformity-only): {predictions['uniformity_only'][i]:.4f}")
    print(f"  Pred (combined): {predictions['combined'][i]:.4f}")

## Bootstrap Confidence Interval

Compute bootstrap confidence interval for the difference in R² between average-only and combined models.

In [ ]:
# Bootstrap CI function - copied from original script
def compute_bootstrap_ci(y_true: np.ndarray, y_pred1: np.ndarray, y_pred2: np.ndarray,
                         n_bootstrap: int = 2000, confidence: float = 0.95) -> dict:
    """
    Compute bootstrap confidence interval for difference in R2 between two models.
    """
    from sklearn.metrics import r2_score
    
    print(f"Computing bootstrap CI with {n_bootstrap} samples...")
    
    n = len(y_true)
    differences = []
    
    for i in range(n_bootstrap):
        if i % 50 == 0 and i > 0:
            print(f"  Bootstrap sample {i}/{n_bootstrap}")
        
        # Sample with replacement
        indices = np.random.choice(n, n, replace=True)
        y_true_b = y_true[indices]
        y_pred1_b = y_pred1[indices]
        y_pred2_b = y_pred2[indices]
        
        # Compute R2 for both
        r2_1 = r2_score(y_true_b, y_pred1_b)
        r2_2 = r2_score(y_true_b, y_pred2_b)
        differences.append(r2_2 - r2_1)
    
    differences = np.array(differences)
    mean_diff = np.mean(differences)
    
    # Confidence interval
    alpha = 1 - confidence
    ci_lower = np.percentile(differences, (alpha/2) * 100)
    ci_upper = np.percentile(differences, (1 - alpha/2) * 100)
    
    # P-value (two-sided test for difference != 0)
    p_value = 2 * min(
        np.mean(differences <= 0),
        np.mean(differences >= 0)
    )
    
    return {
        'mean_diff': float(mean_diff),
        'ci_lower': float(ci_lower),
        'ci_upper': float(ci_upper),
        'p_value': float(p_value),
        'significant': bool(p_value < 0.05)
    }

# Compute bootstrap CI
print("Computing bootstrap CI...")
bootstrap_results = compute_bootstrap_ci(y, predictions['average_only'], predictions['combined'],
                                             n_bootstrap=N_BOOTSTRAP)

print("\n" + "="*60)
print("BOOTSTRAP RESULTS: AVERAGE-ONLY vs COMBINED")
print("="*60)
print(f"Mean difference in R²: {bootstrap_results['mean_diff']:.6f}")
print(f"95% CI: [{bootstrap_results['ci_lower']:.6f}, {bootstrap_results['ci_upper']:.6f}]")
print(f"P-value: {bootstrap_results['p_value']:.6f}")
print(f"Significant: {bootstrap_results['significant']}")

## Results Visualization

Visualize the cross-validation results and compare model performance.

In [ ]:
# Visualization
print("="*60)
print("RESULTS SUMMARY")
print("="*60)
print()

# Print results table
print(f"{'Method':<20} {'R² Mean':<12} {'R² Std':<12}")
print("-"*44)
for method, result in results.items():
    print(f"{method:<20} {result['r2_mean']:<12.4f} {result['r2_std']:<12.4f}")
print()

# Create bar plot of R² scores
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Bar plot
methods = list(results.keys())
r2_means = [results[m]['r2_mean'] for m in methods]
r2_stds = [results[m]['r2_std'] for m in methods]

ax1 = axes[0]
bars = ax1.bar(methods, r2_means, yerr=r2_stds, capsize=10, color=['#3498db', '#e74c3c', '#2ecc71'])
ax1.set_ylabel('R² Score', fontsize=12)
ax1.set_title('Cross-Validation R² Scores by Feature Set', fontsize=14, fontweight='bold')
ax1.set_ylim([min(r2_means) - 0.1, max(r2_means) + 0.1])
ax1.grid(axis='y', alpha=0.3)

# Add value labels on bars
for bar, mean, std in zip(bars, r2_means, r2_stds):
    height = bar.get_height()
    ax1.text(bar.get_x() + bar.get_width()/2., height + std + 0.01,
             f'{mean:.4f}', ha='center', va='bottom', fontweight='bold')

# Bootstrap results visualization
ax2 = axes[1]
bootstrap_data = [
    bootstrap_results['mean_diff'],
    bootstrap_results['ci_lower'],
    bootstrap_results['ci_upper']
]
ax2.bar(['Mean Diff', 'CI Lower', 'CI Upper'], 
        [bootstrap_results['mean_diff'], 
         bootstrap_results['ci_lower'] - bootstrap_results['mean_diff'],
         bootstrap_results['ci_upper'] - bootstrap_results['mean_diff']],
        bottom=[0, bootstrap_results['mean_diff'], bootstrap_results['mean_diff']],
        color=['#9b59b6', '#95a5a6', '#95a5a6'])
ax2.set_ylabel('R² Difference', fontsize=12)
ax2.set_title('Bootstrap CI: Combined - Average-Only', fontsize=14, fontweight='bold')
ax2.axhline(y=0, color='k', linestyle='--', linewidth=0.8)
ax2.grid(axis='y', alpha=0.3)

plt.tight_layout()
plt.show()

# Print bootstrap interpretation
print("\n" + "="*60)
print("INTERPRETATION")
print("="*60)
if bootstrap_results['significant']:
    print("✓ Combined features significantly outperform average-only features")
    print(f"  (p < 0.05, 95% CI does not include 0)")
else:
    print("✗ No significant difference between combined and average-only features")
print()
print("Key insight: Uniformity features (how consistent word properties are")
print("within a sentence) provide additional predictive power beyond")
print("simple averages.")